# Minimal example of Workflow Automation

In [ ]:
from laboneq._automation.logic import FixedParameterUpdate
from laboneq._automation.utils import load_automation_parameters
from laboneq.simple import *

from laboneq_applications._automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.experiments import qubit_spectroscopy
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

qt_platform = demo_platform(n_qubits=6)
setup = qt_platform.setup
qpu = qt_platform.qpu
qubits = qpu.quantum_elements
session = Session(setup)
session.connect(do_emulation=True)

In [ ]:
params_dict = load_automation_parameters("minimal_example.yml")
auto = WorkflowAutomation(session, qpu, automation_parameters=params_dict)

In [ ]:
qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2", "q3"],
    key="qs1",
    depends_on={"root"},
)
auto.add_layer(qs1)
qs2 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2", "q3"],
    key="qs2",
    depends_on={"qs1"},
)
auto.add_layer(qs2)
qs3 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2", "q3"],
    key="qs3",
    depends_on={"qs2"},
)
auto.add_layer(qs3)
auto.plot();

In [ ]:
auto.run_layer("qs1")
auto.plot();

In [ ]:
auto["qs1"].workflow_options

In [ ]:
auto["qs1"].workflow_results[("q0", "q1", "q2", "q3")].tasks[
    "evaluate_experiment"
].output

In [ ]:
auto.reset()
auto.plot();

In [ ]:
auto["qs1"].parameters

In [ ]:
for layer in auto.layers(include_root=False):
    layer.logic = FixedParameterUpdate(
        new_layer_key=layer.key,
        parameter_changes={
            "element_workflow_parameters": {
                "q0": {"evaluation_fit_r2_thresholds": -0.0003},
                "q1": {"evaluation_fit_r2_thresholds": -0.0003},
                "q2": {"evaluation_fit_r2_thresholds": -0.0003},
                "q3": {"evaluation_fit_r2_thresholds": -0.0003},
            }
        },
    )

In [ ]:
auto.run()
auto.plot();

In [ ]:
auto["qs1"].parameters

In [ ]:
auto["qs1"].fail_count

In [ ]:
auto["qs1"].pass_count

In [ ]:
auto["qs1"].workflow_results

In [ ]:
auto.reset()
auto.plot();

In [ ]:
auto.run_from_node("qs1_q0")
auto.plot();

In [ ]:
auto["qs3"].workflow_results